# MotoGP Constructor World Championship Analysis

This notebook explores the MotoGP constructor championship data from 1949-2022.

## 0. Notebook Setup

In [ ]:
# Import necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Configure plot styles
plt.style.use('default')
sns.set_palette("husl")

## 1. Data Loading

Let's load the CSV file with MotoGP constructor championship data.

In [3]:
# Load data from CSV
data_path = Path("../data/raw/constructure_world_championship.csv")
df = pd.read_csv(data_path)

print(f"Data loaded successfully!")

Data loaded successfully!


In [4]:
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

Shape: (284, 3)

First 5 rows:


,Season,Constructor,Class
0,2021,Yamaha,MotoGP™
1,2021,Kalex,Moto2™
2,2021,KTM,Moto3™
3,2021,Energica,MotoE™
4,2020,Suzuki,MotoGP™


## 2. Data Structure Exploration

Let's examine the data structure and check basic information about the dataset.

### General dataset information

In [7]:
print(f"Dimensions: {df.shape}")
print("---------------------------------")
print(f"Columns: {list(df.columns)}")
print("---------------------------------")
print(f"\nData types:")
print(df.dtypes)
print("---------------------------------")
print(f"\nNull values:")
print(df.isnull().sum())

Dimensions: (284, 3)
---------------------------------
Columns: ['Season', 'Constructor', 'Class']
---------------------------------

Data types:
Season          int64
Constructor    object
Class          object
dtype: object
---------------------------------

Null values:
Season         0
Constructor    0
Class          0
dtype: int64


In [ ]:
# Unique values in each column
print("=== UNIQUE VALUES ===")
for col in df.columns:
    unique_count = df[col].nunique()
    print(f"\n{col}: {unique_count} unique values")
    if unique_count <= 20:  # Show values if there are few
        print(f"Values: {sorted(df[col].unique())}")
    else:
        print(f"Sample: {sorted(df[col].unique())[:10]}...")

## 3. Descriptive Analysis

Let's analyze the data distribution by constructor, class and time period.

In [ ]:
# Analysis by Constructor
print("=== ANALYSIS BY CONSTRUCTOR ===")
constructor_counts = df['Constructor'].value_counts()
print("\nNumber of titles per constructor:")
print(constructor_counts)

print(f"\nData period:")
print(f"Initial year: {df['Season'].min()}")
print(f"Final year: {df['Season'].max()}")
print(f"Total years: {df['Season'].max() - df['Season'].min() + 1}")

In [ ]:
# Analysis by Class
print("=== ANALYSIS BY CLASS ===")
class_counts = df['Class'].value_counts()
print("\nNumber of titles per class:")
print(class_counts)

# Combined analysis
print("\n=== COMBINED ANALYSIS ===")
class_constructor = df.groupby(['Class', 'Constructor']).size().unstack(fill_value=0)
print("\nTitles per Constructor and Class:")
print(class_constructor)

## 4. Visualizations

Let's create charts to better understand the distribution of titles.

In [ ]:
# Bar chart: Titles per Constructor
plt.figure(figsize=(12, 6))
constructor_counts = df['Constructor'].value_counts()
constructor_counts.plot(kind='bar', color='skyblue')
plt.title('Number of Titles per Constructor')
plt.xlabel('Constructor')
plt.ylabel('Number of Titles')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Pie chart: Distribution by Class
plt.figure(figsize=(8, 8))
class_counts = df['Class'].value_counts()
plt.pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Titles by Class')
plt.axis('equal')
plt.show()

In [ ]:
# Heatmap: Titles by Constructor and Class
plt.figure(figsize=(12, 8))
class_constructor_pivot = df.groupby(['Class', 'Constructor']).size().unstack(fill_value=0)
sns.heatmap(class_constructor_pivot, annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': 'Number of Titles'})
plt.title('Titles by Constructor and Class')
plt.xlabel('Constructor')
plt.ylabel('Class')
plt.tight_layout()
plt.show()

In [ ]:
# Timeline: Titles over the years by Constructor (top 5)
top_constructors = df['Constructor'].value_counts().head(5).index
df_top = df[df['Constructor'].isin(top_constructors)]

plt.figure(figsize=(14, 8))
for constructor in top_constructors:
    constructor_data = df_top[df_top['Constructor'] == constructor]
    years = constructor_data['Season'].value_counts().sort_index()
    plt.plot(years.index, years.cumsum(), marker='o', label=constructor, linewidth=2)

plt.title('Cumulative Evolution of Titles - Top 5 Constructors')
plt.xlabel('Year')
plt.ylabel('Cumulative Titles')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Insights and Patterns

Let's identify interesting patterns in the data.

In [ ]:
# Dominance analysis by decades
df['Decade'] = (df['Season'] // 10) * 10
decade_analysis = df.groupby(['Decade', 'Constructor']).size().unstack(fill_value=0)

print("=== DOMINANCE BY DECADES ===")
for decade in sorted(df['Decade'].unique()):
    decade_data = df[df['Decade'] == decade]
    top_constructor = decade_data['Constructor'].value_counts().index[0]
    count = decade_data['Constructor'].value_counts().iloc[0]
    total = len(decade_data)
    percentage = (count / total) * 100
    print(f"{decade}s: {top_constructor} ({count}/{total} titles, {percentage:.1f}%)")

In [ ]:
# Most versatile constructors (winning in multiple classes)
print("=== MOST VERSATILE CONSTRUCTORS ===")
constructor_classes = df.groupby('Constructor')['Class'].nunique().sort_values(ascending=False)
versatile_constructors = constructor_classes[constructor_classes > 1]

print("Constructors that won in multiple classes:")
for constructor, num_classes in versatile_constructors.items():
    classes = df[df['Constructor'] == constructor]['Class'].unique()
    print(f"{constructor}: {num_classes} classes ({', '.join(classes)})")

In [ ]:
# Consecutive title sequences
print("=== CONSECUTIVE TITLE SEQUENCES ===")

# Analyze by class
for class_name in df['Class'].unique():
    class_data = df[df['Class'] == class_name].sort_values('Season')
    print(f"\n{class_name}:")
    
    current_constructor = None
    current_streak = 0
    max_streak = 0
    max_streak_constructor = None
    
    for _, row in class_data.iterrows():
        if row['Constructor'] == current_constructor:
            current_streak += 1
        else:
            if current_streak > max_streak:
                max_streak = current_streak
                max_streak_constructor = current_constructor
            current_constructor = row['Constructor']
            current_streak = 1
    
    # Check last sequence
    if current_streak > max_streak:
        max_streak = current_streak
        max_streak_constructor = current_constructor
    
    if max_streak > 1:
        print(f"  Longest streak: {max_streak_constructor} ({max_streak} consecutive titles)")

## 6. Summary of Key Insights

Based on the analysis performed, we can highlight the following key points about MotoGP constructor champions:

### Leading Constructors
- **Honda**: Historically leads with the most titles
- **Yamaha**: Strong presence especially in the premier category
- **Suzuki**: Consistent competitor throughout the decades

### Distribution by Classes
- The analysis shows the evolution of different classes over the years
- Some constructors specialized in specific classes
- Others demonstrated versatility by winning in multiple categories

### Temporal Patterns
- Different decades were dominated by different constructors
- Certain brands had periods of prolonged dominance
- Competitiveness varied significantly over the years

### Next Steps
This notebook provides a solid foundation for more in-depth analyses, such as:
- Correlation with rider data
- Analysis of circuits where each constructor is strongest
- Technological evolution and its impact on results